# 🏨 Agentic Hotel Assistant — Notebook 1: Setup
## RAG (Hotel Documents) + SQL Database + Tools

This notebook builds the foundation:
1. Install dependencies
2. Create fake hotel documents → build a FAISS vector store (RAG)
3. Create a fake SQLite database with rooms & bookings
4. Define LangChain tools the agent will use

## 1. Install Dependencies

In [ ]:
# !pip install langchain langchain-community langchain-openai langgraph faiss-cpu openai tiktoken python-dotenv -q

## 2. Environment Setup

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # Load from .env file if exists

# Set your OpenAI API key here OR put it in a .env file as OPENAI_API_KEY=...
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "sk-YOUR-KEY-HERE")

print("✅ Environment ready.")

✅ Environment ready.


## 3. Create Fake Hotel Documents (for RAG Knowledge Base)

These documents simulate what a real hotel would store in its knowledge base — policies, amenities, restaurant info, etc.

In [2]:
# ─────────────────────────────────────────────────────────────
# HOTEL DOCUMENT CORPUS  (simulated knowledge base)
# ─────────────────────────────────────────────────────────────

hotel_documents = [
    # ── General Info
    """
    GRAND AZURE HOTEL — OVERVIEW
    
    Grand Azure Hotel is a 5-star luxury hotel located in the heart of Cairo, Egypt.
    It was established in 1998 and has been a landmark of Egyptian hospitality ever since.
    The hotel has 12 floors and 200 rooms of various categories.
    Our address is: 15 Nile Corniche Street, Cairo, Egypt.
    Phone: +20-2-2345-6789 | Email: info@grandazure.com
    Check-in time: 2:00 PM | Check-out time: 12:00 PM (noon)
    Early check-in and late check-out are available upon request and subject to availability.
    """,

    # ── Room Types
    """
    ROOM TYPES AND PRICING — GRAND AZURE HOTEL

    1. Standard Room:
       - Size: 28 sqm
       - Beds: 1 Queen bed or 2 Twin beds
       - Price: $80 per night
       - Features: Air conditioning, flat-screen TV, mini-fridge, free Wi-Fi, work desk

    2. Deluxe Room:
       - Size: 38 sqm
       - Beds: 1 King bed or 2 Queen beds
       - Price: $130 per night
       - Features: All Standard features + Nile view, bathtub, sitting area, premium toiletries

    3. Suite:
       - Size: 65 sqm
       - Beds: 1 King bed
       - Price: $220 per night
       - Features: Separate living room, kitchenette, butler service, panoramic Nile view, jacuzzi

    4. Presidential Suite:
       - Size: 120 sqm
       - Beds: 1 King bed + 1 sofa bed
       - Price: $450 per night
       - Features: Private terrace, dining area for 6, personal butler 24/7, private gym access

    All rooms include: Free Wi-Fi, daily housekeeping, complimentary bottled water,
    in-room safe, hair dryer, and iron/ironing board.
    """,

    # ── Amenities
    """
    HOTEL AMENITIES — GRAND AZURE HOTEL

    SWIMMING POOL:
    - Outdoor infinity pool on the 10th floor with Nile views
    - Open daily from 7:00 AM to 10:00 PM
    - Pool bar service available 9:00 AM – 9:00 PM
    - Children's pool available (ages 3–12)
    - Towels provided free of charge

    FITNESS CENTER:
    - Fully equipped gym on floor 9
    - Open 24 hours for hotel guests
    - Personal trainers available (book at front desk, $30/session)
    - Equipment: Treadmills, ellipticals, free weights, resistance machines, yoga mats

    SPA & WELLNESS:
    - Azure Spa located on floor 8
    - Services: Swedish massage, deep tissue massage, aromatherapy, facials, hammam
    - Operating hours: 9:00 AM – 9:00 PM daily
    - Advance booking required (call ext. 800 or front desk)
    - Couple packages available starting at $120

    BUSINESS CENTER:
    - Located on floor 2
    - Services: Printing, scanning, faxing, meeting rooms
    - Meeting rooms available for rent: Small (10 pax, $50/hr), Large (30 pax, $100/hr)
    - High-speed fiber internet throughout the hotel

    OTHER FACILITIES:
    - Concierge services: Tour booking, airport transfers, car rentals
    - Laundry and dry-cleaning service (same-day available before 9 AM)
    - Gift shop on floor 1 (open 8 AM – 11 PM)
    - Pharmacy on-call 24/7
    - Valet parking available ($15/day)
    - Free self-parking also available
    """,

    # ── Restaurants
    """
    DINING & RESTAURANTS — GRAND AZURE HOTEL

    1. THE NILE GRILL (Floor 1 – Main Restaurant)
       - Cuisine: International Buffet and À la carte
       - Breakfast Buffet: 6:30 AM – 10:30 AM ($25/person, free for Suite and above)
       - Lunch: 12:30 PM – 3:30 PM
       - Dinner: 7:00 PM – 11:00 PM
       - Dress code: Smart casual
       - Reservations recommended for dinner (ext. 810)

    2. AZURE ROOFTOP LOUNGE (Floor 12)
       - Cuisine: Light bites, cocktails, mocktails, afternoon tea
       - Hours: 4:00 PM – 1:00 AM
       - Stunning 360° views of Cairo and the Nile
       - Live music: Friday & Saturday evenings from 8:00 PM
       - Minimum spend: $20/person on weekends

    3. CAIRO KITCHEN (Floor 1 – Casual Dining)
       - Cuisine: Traditional Egyptian and Middle Eastern food
       - Hours: 11:00 AM – 11:00 PM
       - Signature dishes: Koshary, Feteer Meshaltet, Grilled Kofta, Fattah
       - Family-friendly atmosphere

    4. SUSHI NILE (Floor 3)
       - Cuisine: Japanese and Pan-Asian
       - Hours: 1:00 PM – 11:00 PM (Closed Mondays)
       - Signature dishes: Dragon Roll, Black Miso Ramen, Wagyu Beef Sashimi
       - Reservations highly recommended (ext. 830)

    IN-ROOM DINING:
       - Available 24/7
       - Full menu available from 7 AM to 11 PM
       - Limited menu (snacks, sandwiches, beverages) 11 PM – 7 AM
       - Delivery time: approximately 30 minutes
       - Call ext. 850 or use the in-room tablet
    """,

    # ── Policies
    """
    HOTEL POLICIES — GRAND AZURE HOTEL

    CANCELLATION POLICY:
    - Free cancellation up to 48 hours before check-in date
    - Cancellations within 48 hours: 1 night charge applies
    - No-show: Full stay charged
    - Non-refundable rates: Cannot be cancelled or modified

    PAYMENT:
    - Accepted: Visa, MasterCard, American Express, cash (EGP or USD)
    - A credit card authorization hold of $100 is taken at check-in for incidentals
    - Hold is released within 3–5 business days after checkout

    PET POLICY:
    - Small pets (under 10 kg) are welcome
    - Pet fee: $30/night
    - Pets must not be left alone in rooms unattended
    - Available: Pet beds, bowls (request at front desk)

    SMOKING POLICY:
    - All indoor areas are strictly non-smoking
    - Designated smoking areas available on floors 3 and 7 (outdoor balconies)
    - Smoking in rooms results in a $200 cleaning fee

    CHILDREN:
    - Children under 12 stay free when using existing bedding
    - Extra bed/crib: $20/night (request at time of booking)
    - Kids' menu available in all restaurants
    """,

    # ── Location & Transport
    """
    LOCATION AND TRANSPORTATION — GRAND AZURE HOTEL

    LOCATION:
    Grand Azure Hotel is located on the Nile Corniche in central Cairo,
    within walking distance of major landmarks:
    - Egyptian Museum: 1.2 km (15-minute walk)
    - Tahrir Square: 0.8 km (10-minute walk)
    - Cairo Tower: 3 km (10-minute drive)
    - Khan El-Khalili Bazaar: 4 km (15-minute drive)

    AIRPORT TRANSFERS:
    - Cairo International Airport is 30 km away (approximately 45 min by car)
    - Private car transfer: $40 one-way (book at concierge)
    - Shared shuttle: $15/person (departs at fixed times: 6 AM, 10 AM, 2 PM, 6 PM)

    CAR RENTAL:
    - Available through front desk (partner agencies)
    - Starting from $45/day

    METRO:
    - Nearest metro station: Sadat Station (Cairo Metro Line 1 & 2) — 5-minute walk

    TOURS:
    - Day trips to Pyramids of Giza: $60/person (includes guide and transport)
    - Nile dinner cruise: $55/person (departs 8 PM, book 1 day in advance)
    - Alexandria day trip: $80/person
    """
]

print(f"✅ Created {len(hotel_documents)} hotel knowledge documents.")

✅ Created 6 hotel knowledge documents.


## 4. Build the RAG Vector Store (FAISS)

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document

# Step 1: Wrap strings as LangChain Documents
docs = [Document(page_content=text.strip()) for text in hotel_documents]

# Step 2: Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(docs)
print(f"📄 Total chunks after splitting: {len(chunks)}")

# Step 3: Embed and store in FAISS
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)

# Step 4: Save to disk so Notebook 2 can load it
vectorstore.save_local("hotel_vectorstore")
print("✅ FAISS vector store saved to 'hotel_vectorstore/'")

📄 Total chunks after splitting: 22
✅ FAISS vector store saved to 'hotel_vectorstore/'


### Quick RAG Test

In [8]:
# Quick retrieval test
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
results = retriever.invoke("What restaurants does the hotel have?")
print("🔍 Test query: 'What restaurants does the hotel have?'")
print(f"   Retrieved {len(results)} chunks. Preview of first result:")
print(results[0].page_content[:300])

🔍 Test query: 'What restaurants does the hotel have?'
   Retrieved 3 chunks. Preview of first result:
IN-ROOM DINING:
       - Available 24/7
       - Full menu available from 7 AM to 11 PM
       - Limited menu (snacks, sandwiches, beverages) 11 PM – 7 AM
       - Delivery time: approximately 30 minutes
       - Call ext. 850 or use the in-room tablet


## 5. Create the SQLite Database (Hotel Booking System)

In [9]:
import sqlite3
import pandas as pd
from datetime import date

conn = sqlite3.connect("hotel.db")
cursor = conn.cursor()

# ── Table 1: rooms
cursor.execute("DROP TABLE IF EXISTS rooms")
cursor.execute("""
CREATE TABLE rooms (
    room_id     INTEGER PRIMARY KEY,
    room_number TEXT NOT NULL,
    room_type   TEXT NOT NULL,   -- Standard, Deluxe, Suite, Presidential
    floor       INTEGER,
    price_per_night REAL,
    is_available INTEGER DEFAULT 1  -- 1=available, 0=booked
)
""")

rooms_data = [
    # Standard rooms (floors 2-5)
    (1, '201', 'Standard', 2, 80.0, 1),
    (2, '202', 'Standard', 2, 80.0, 1),
    (3, '203', 'Standard', 2, 80.0, 0),  # booked
    (4, '301', 'Standard', 3, 80.0, 1),
    (5, '302', 'Standard', 3, 80.0, 0),  # booked
    (6, '401', 'Standard', 4, 80.0, 1),
    (7, '501', 'Standard', 5, 80.0, 1),
    # Deluxe rooms (floors 6-7)
    (8,  '601', 'Deluxe', 6, 130.0, 1),
    (9,  '602', 'Deluxe', 6, 130.0, 0),  # booked
    (10, '603', 'Deluxe', 6, 130.0, 1),
    (11, '701', 'Deluxe', 7, 130.0, 1),
    (12, '702', 'Deluxe', 7, 130.0, 1),
    # Suites (floors 8-9)
    (13, '801', 'Suite', 8, 220.0, 1),
    (14, '802', 'Suite', 8, 220.0, 0),  # booked
    (15, '901', 'Suite', 9, 220.0, 1),
    # Presidential (floor 11-12)
    (16, '1101', 'Presidential', 11, 450.0, 1),
    (17, '1201', 'Presidential', 12, 450.0, 0),  # booked
]
cursor.executemany("INSERT INTO rooms VALUES (?,?,?,?,?,?)", rooms_data)

# ── Table 2: bookings
cursor.execute("DROP TABLE IF EXISTS bookings")
cursor.execute("""
CREATE TABLE bookings (
    booking_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    guest_name   TEXT NOT NULL,
    room_id      INTEGER,
    check_in     TEXT,   -- YYYY-MM-DD
    check_out    TEXT,   -- YYYY-MM-DD
    total_price  REAL,
    status       TEXT DEFAULT 'confirmed',  -- confirmed, cancelled
    FOREIGN KEY (room_id) REFERENCES rooms(room_id)
)
""")

bookings_data = [
    ('Ahmed Hassan',  3,  '2025-07-10', '2025-07-13', 240.0,  'confirmed'),
    ('Fatma Ali',     5,  '2025-07-15', '2025-07-17', 160.0,  'confirmed'),
    ('John Smith',    9,  '2025-07-12', '2025-07-15', 390.0,  'confirmed'),
    ('Sara Mohamed',  14, '2025-07-20', '2025-07-22', 440.0,  'confirmed'),
    ('Khalid Omar',   17, '2025-07-08', '2025-07-11', 1350.0, 'confirmed'),
    ('Layla Ibrahim',  2, '2025-07-01', '2025-07-03', 160.0,  'confirmed'),
]
cursor.executemany(
    "INSERT INTO bookings (guest_name,room_id,check_in,check_out,total_price,status) VALUES (?,?,?,?,?,?)",
    bookings_data
)

conn.commit()
conn.close()
print("✅ hotel.db created with 'rooms' and 'bookings' tables.")

✅ hotel.db created with 'rooms' and 'bookings' tables.


In [10]:
# ── Inspect the database
conn = sqlite3.connect("hotel.db")

print("=== ROOMS ===")
print(pd.read_sql("SELECT * FROM rooms", conn).to_string(index=False))

print("\n=== BOOKINGS ===")
print(pd.read_sql("SELECT * FROM bookings", conn).to_string(index=False))

conn.close()

=== ROOMS ===
 room_id room_number    room_type  floor  price_per_night  is_available
       1         201     Standard      2             80.0             1
       2         202     Standard      2             80.0             1
       3         203     Standard      2             80.0             0
       4         301     Standard      3             80.0             1
       5         302     Standard      3             80.0             0
       6         401     Standard      4             80.0             1
       7         501     Standard      5             80.0             1
       8         601       Deluxe      6            130.0             1
       9         602       Deluxe      6            130.0             0
      10         603       Deluxe      6            130.0             1
      11         701       Deluxe      7            130.0             1
      12         702       Deluxe      7            130.0             1
      13         801        Suite      8          

## 6. Define the Agent Tools

We define **4 tools**:
| Tool | Purpose |
|---|---|
| `search_hotel_info` | RAG — search hotel documents |
| `check_room_availability` | SQL — check which rooms are free |
| `get_booking_details` | SQL — look up a booking by guest name |
| `book_room` | SQL — create a new booking |

In [11]:
import sqlite3
from langchain.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# ── Load vector store (for RAG tool)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.load_local(
    "hotel_vectorstore",
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

DB_PATH = "hotel.db"


# ─────────────────────────────────────────
# TOOL 1 — RAG: Search Hotel Information
# ─────────────────────────────────────────
@tool
def search_hotel_info(query: str) -> str:
    """Search the hotel knowledge base for information about amenities, 
    rooms, restaurants, policies, location, and hotel services.
    Use this for any general informational question about the hotel."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information found in the hotel knowledge base."
    results = "\n\n---\n\n".join(d.page_content for d in docs)
    return f"Hotel Knowledge Base Results:\n\n{results}"


# ─────────────────────────────────────────
# TOOL 2 — SQL: Check Room Availability
# ─────────────────────────────────────────
@tool
def check_room_availability(room_type: str = "") -> str:
    """Check which hotel rooms are currently available.
    Optionally filter by room type: Standard, Deluxe, Suite, or Presidential.
    Leave room_type empty to see all available rooms."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    if room_type.strip():
        cursor.execute(
            "SELECT room_number, room_type, floor, price_per_night FROM rooms "
            "WHERE is_available=1 AND LOWER(room_type)=LOWER(?)",
            (room_type,)
        )
    else:
        cursor.execute(
            "SELECT room_number, room_type, floor, price_per_night FROM rooms WHERE is_available=1"
        )

    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return f"No available rooms found" + (f" of type '{room_type}'." if room_type else ".")

    header = "Room No | Type           | Floor | Price/Night\n" + "-"*50
    lines  = [f"{r[0]:<8} | {r[1]:<14} | {r[2]:<5} | ${r[3]:.2f}" for r in rows]
    return f"Available Rooms:\n{header}\n" + "\n".join(lines)


# ─────────────────────────────────────────
# TOOL 3 — SQL: Get Booking Details
# ─────────────────────────────────────────
@tool
def get_booking_details(guest_name: str) -> str:
    """Look up an existing booking by the guest's name.
    Returns booking details including room, dates, and total price."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        """
        SELECT b.booking_id, b.guest_name, r.room_number, r.room_type,
               b.check_in, b.check_out, b.total_price, b.status
        FROM bookings b
        JOIN rooms r ON b.room_id = r.room_id
        WHERE LOWER(b.guest_name) LIKE LOWER(?)
        """,
        (f"%{guest_name}%",)
    )
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return f"No booking found for guest '{guest_name}'."

    output = []
    for r in rows:
        output.append(
            f"Booking ID   : {r[0]}\n"
            f"Guest Name   : {r[1]}\n"
            f"Room         : {r[2]} ({r[3]})\n"
            f"Check-In     : {r[4]}\n"
            f"Check-Out    : {r[5]}\n"
            f"Total Price  : ${r[6]:.2f}\n"
            f"Status       : {r[7]}"
        )
    return "\n\n".join(output)


# ─────────────────────────────────────────
# TOOL 4 — SQL: Book a Room
# ─────────────────────────────────────────
@tool
def book_room(guest_name: str, room_number: str, check_in: str, check_out: str) -> str:
    """Book a hotel room for a guest.
    Args:
        guest_name: Full name of the guest
        room_number: Room number to book (e.g. '201', '601', '801')
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
    Returns confirmation or error message."""
    from datetime import datetime

    # Validate dates
    try:
        ci = datetime.strptime(check_in, "%Y-%m-%d")
        co = datetime.strptime(check_out, "%Y-%m-%d")
        if co <= ci:
            return "Error: Check-out date must be after check-in date."
        nights = (co - ci).days
    except ValueError:
        return "Error: Dates must be in YYYY-MM-DD format."

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Check room exists and is available
    cursor.execute(
        "SELECT room_id, room_type, price_per_night, is_available FROM rooms WHERE room_number=?",
        (room_number,)
    )
    room = cursor.fetchone()

    if not room:
        conn.close()
        return f"Error: Room {room_number} does not exist."

    room_id, room_type, price, is_available = room

    if not is_available:
        conn.close()
        return f"Error: Room {room_number} is not available. Please choose another room."

    total = price * nights

    # Insert booking
    cursor.execute(
        "INSERT INTO bookings (guest_name, room_id, check_in, check_out, total_price, status) VALUES (?,?,?,?,?,'confirmed')",
        (guest_name, room_id, check_in, check_out, total)
    )
    booking_id = cursor.lastrowid

    # Mark room as unavailable
    cursor.execute("UPDATE rooms SET is_available=0 WHERE room_id=?", (room_id,))

    conn.commit()
    conn.close()

    return (
        f"✅ Booking Confirmed!\n"
        f"Booking ID  : {booking_id}\n"
        f"Guest       : {guest_name}\n"
        f"Room        : {room_number} ({room_type})\n"
        f"Check-In    : {check_in}\n"
        f"Check-Out   : {check_out}\n"
        f"Nights      : {nights}\n"
        f"Total Price : ${total:.2f}\n"
        f"Status      : Confirmed"
    )


# Bundle all tools
tools = [search_hotel_info, check_room_availability, get_booking_details, book_room]

print(f"✅ {len(tools)} tools defined: {[t.name for t in tools]}")

✅ 4 tools defined: ['search_hotel_info', 'check_room_availability', 'get_booking_details', 'book_room']


### Quick Tool Tests

In [12]:
# Test RAG tool
print(search_hotel_info.invoke("What is the spa schedule?")[:400])

Hotel Knowledge Base Results:

SPA & WELLNESS:
    - Azure Spa located on floor 8
    - Services: Swedish massage, deep tissue massage, aromatherapy, facials, hammam
    - Operating hours: 9:00 AM – 9:00 PM daily
    - Advance booking required (call ext. 800 or front desk)
    - Couple packages available starting at $120

---

FITNESS CENTER:
    - Fully equipped gym on floor 9
    - Open 24 hours


In [13]:
# Test availability tool
print(check_room_availability.invoke("Deluxe"))

Available Rooms:
Room No | Type           | Floor | Price/Night
--------------------------------------------------
601      | Deluxe         | 6     | $130.00
603      | Deluxe         | 6     | $130.00
701      | Deluxe         | 7     | $130.00
702      | Deluxe         | 7     | $130.00


In [14]:
# Test booking lookup
print(get_booking_details.invoke("Ahmed"))

Booking ID   : 1
Guest Name   : Ahmed Hassan
Room         : 203 (Standard)
Check-In     : 2025-07-10
Check-Out    : 2025-07-13
Total Price  : $240.00
Status       : confirmed


In [15]:
# Test booking creation
print(book_room.invoke({
    "guest_name": "Test Guest",
    "room_number": "401",
    "check_in": "2025-08-01",
    "check_out": "2025-08-03"
}))

✅ Booking Confirmed!
Booking ID  : 7
Guest       : Test Guest
Room        : 401 (Standard)
Check-In    : 2025-08-01
Check-Out   : 2025-08-03
Nights      : 2
Total Price : $160.00
Status      : Confirmed


---
## ✅ Notebook 1 Complete!

What we built:
- 📚 **RAG**: 6 hotel documents → FAISS vector store saved to `hotel_vectorstore/`
- 🗄️ **SQL**: SQLite database `hotel.db` with `rooms` and `bookings` tables
- 🔧 **4 Tools**: RAG search, availability check, booking lookup, room booking

➡️ **Now open Notebook 2** to build the LangGraph agent!